# Synchronous Loop with Ophyd and Ophyd-Async

## Imports

### Global Imports

In [1]:
import numpy as np

from bluesky import RunEngine
from bluesky.callbacks import LiveTable
from bluesky.plans import count, list_scan
import bluesky.plan_stubs as bps

from ophyd_async.core import init_devices

### Local-File Imports

In [2]:
from sidekick_model3_CA_devices import PulseGenerator, LilLaser, Diode
from sidekick_model3_PVA_devices_v3 import DiodePVA

## Initialize our Sidekick Model 3 Devices

### Simple Device Interfaces (Channel Access, ophyd)

In [3]:
pulsegen = PulseGenerator('PULSEGEN:', name='pulsegen')
laser = LilLaser('LASER:', name='laser')
electron = Diode('ELECTRON:', name='electron')
proton = Diode('PROTON:', name='proton')

for device in [pulsegen, laser, electron, proton]:
    device.wait_for_connection()

### High-Performance Device Interfaces (PVAccess, ophyd-async)

#### Start Bluesky RunEngine (before initializing ophyd-async devices)

In [4]:
RE = RunEngine()

#### Initialize ophyd-async devices

In [5]:
with init_devices():
    electron_pva = DiodePVA(prefix="pva://ELECTRON-DAQ:", name="electron_pva")
    proton_pva = DiodePVA(prefix="pva://PROTON-DAQ:", name="proton_pva")

## Create Plan to Put Sidekick in a Known Initial State

Set timing settings and rep-rate.

In [18]:
# Written by ChatGPT with help from Scott Feister on 2026-06-22.
# Simple Bluesky pre-run setup helper for putting Sidekick devices
# into a known initial state before opening a run.

def prepare_for_run():
    """Put the Sidekick Model 3 into the standard initial state before a run.
    """

    # Set all delta-t values.
    yield from bps.mv(
        proton.dt, 5.0e-6,        # seconds
        electron.dt, 5.0e-6,      # seconds
        laser.powers_dt, 5.0,     # microseconds
    )

    # Set trigger delays to known values.
    yield from bps.mv(
        pulsegen.ch2_delay, 100.0,    # microseconds; proton delay
        pulsegen.ch3_delay, 100.0,    # microseconds; electron delay
        pulsegen.ch4_delay, 100.0,    # microseconds; laser delay
    )

    # Set system repetition rate.
    yield from bps.mv(
        pulsegen.reprate, 50.0,       # Hz
    )

### Execute this Plan

In [13]:
RE(prepare_for_run())

()

## Create plan to Set Laser Powers and Return Traces

In [34]:
def one_pulse(pulse, md=None):
    # Data validity checks
    assert(isinstance(pulse, np.ndarray))
    assert(np.ndim(pulse) == 1)
    assert(len(pulse) == 100)
    assert(np.max(pulse) <= 255)
    assert(np.min(pulse) >= 0)
    if (np.count_nonzero(pulse - pulse.round() > 0)):
        raise Exception("Pulse array contains fractional values (should be only integers, from 0 to 255).")

    # Data type conversion to uint8
    pulse_uint8 = np.uint8(pulse)

    
    yield from bps.open_run(md=md)
    
    yield from bps.mv(
        laser.powers, pulse_uint8,
    )

    reading = yield from bps.trigger_and_read([
        laser.powers,
        electron_pva.trace,
        proton_pva.trace,
    ])

    yield from bps.close_run()

    return reading

In [35]:
pulse_flat = np.round(np.ones(100)*255)
RE(one_pulse(pulse_flat))

('53b8d87c-db61-46fb-b552-0c014aef2047',)